# ColdCap: kratší přemýšlení Qwen3-0.6B na Kaggle

Nastavení notebooku (pravý panel): **Accelerator = GPU T4 x2**, **Internet = On**.

Postup:
1. setup (klon repa, instalace, data, otázky evaluace),
2. trénink `train_coldcap.py` (GPU 0 trénuje, GPU 1 generuje přes vLLM),
3. evaluace `eval_pair.py` (baseline a model na **stejné kartě a ve stejné přesnosti**).

Kaggle maže `/kaggle/working` po konci session. Dlouhé běhy proto pouštěj přes **Save Version → Save & Run All**: výstup se uloží a další session ho může připojit jako input a pokračovat přes `--resume`.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

In [ ]:
REPO_URL = "https://github.com/vbtronic/ThinkingCap-Test.git"   # tvoje repo
BRANCH = "main"
WORK = "/kaggle/working"
REPO = f"{WORK}/ThinkingCap-Test"
OUT = f"{WORK}/outputs/run1"

import os
if not os.path.exists(REPO):
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
# vLLM a pevné verze transformers/peft podle requirements.txt (několik minut)
!pip install -q uv
!uv pip install --system -q -r requirements.txt

In [ ]:
!python utils/train/prepare_data.py
!python utils/eval/benchmarks.py
!python -m coldcap.hw

## Pokračování po odpojení (volitelné)

Pokud máš výstup z minulé verze notebooku připojený jako input (Add Input → Your Work), zkopíruj ho sem a v tréninku nech `--resume`.

In [ ]:
PREVIOUS = ""   # např. "/kaggle/input/coldcap-train/outputs/run1"
if PREVIOUS and not os.path.exists(OUT):
    !mkdir -p {WORK}/outputs && cp -r {PREVIOUS} {OUT}

## Trénink

Na začátek stačí krátký běh (`--steps 20`), ať je vidět, že vše funguje. Pak plný běh.
Každých `--eval-every` kroků se uloží checkpoint `ckpt-XXXXX` s čísly; `curve.json` je křivka délka vs. přesnost.

In [ ]:
!python train_coldcap.py --out {OUT} --resume \
    --train-gpu 0 --rollout-gpu 1 \
    --steps 200 --eval-every 20 \
    --prompts-per-step 16 --group 6

In [ ]:
import json, pathlib
for row in json.loads(pathlib.Path(OUT, "curve.json").read_text()):
    print(row["checkpoint"], f"saved {row['saved_pct']:+.1f}%", f"drop {row['drop_pp']:+.2f} pp",
          "OK" if row["within_budget"] else "over budget")

## Evaluace

`eval_pair.py` změří baseline na **té samé kartě a dtype** jako tvůj model (a uloží ji do `runs/base/<karta>-<dtype>-...`, takže se měří jen jednou), pak tvůj model, pak porovná otázku po otázce.
Kartu vybírá `CUDA_VISIBLE_DEVICES`. Nejdřív smoke test (~minuta), pak rychlá kontrola na SVAMP, pak plná evaluace (~5 h na T4 pro každý z dvou běhů – na samostatnou session).

In [ ]:
CKPT = f"{OUT}/latest"   # nebo konkrétní f"{OUT}/ckpt-00100"
!CUDA_VISIBLE_DEVICES=0 python eval_pair.py --adapter {CKPT} --out runs/smoke --smoke

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python eval_pair.py --adapter {CKPT} --out runs/svamp-s2 --benchmarks svamp --samples 2

In [ ]:
# Plná evaluace (8 vzorků, všechny 4 benchmarky) -- dlouhé, pouštěj přes Save & Run All
# !CUDA_VISIBLE_DEVICES=0 python eval_pair.py --adapter {CKPT} --out runs/full

In [ ]:
# Kalibrace karet (A/A): jak moc se liší samotný hardware.
# Potřebuje baseline změřenou na dvou různých setupech, např. GPU 0 vs GPU 1 nebo T4 vs jiná karta:
# !python compare.py runs/base/<setup-A> runs/base/<setup-B>
!ls runs/base 2>/dev/null